# PCA30 GMM cluster-count and covariance comparison

Reuse the exact training-only PCA30 representation from the previous experiment. Compare **K=10,20,30** with **tied and diagonal covariance**: six configurations, including the previously evaluated K=10/tied control. Keep all other GMM initialization/convergence/regularization settings and partial-label acceptance/weight rules unchanged.

The final classifier remains one-vs-rest logistic regression on all **150 original embeddings**, with its frozen scaler, C=1000, fitting-weight total 157 per reason, cutoff 0.5, and highest-probability fallback. Changes in clusters may change generated targets/weights, but known labels and their raw weight 1 are preserved. No new features, validation/test fitting, final train+validation refit, self-training, Kaggle submission, commit, or tag.

Compare against the PCA30/tied/K10 control (sample F1 0.897436), original no-PCA tied/K10 GMM (0.905983), and current DBSCAN leader (0.910256). This tests the larger-cluster/diagonal combination under our pipeline; it does not copy Ani's full transductive/self-training model.

In [1]:
from pathlib import Path
import hashlib
import json
import platform
import warnings
import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from threadpoolctl import threadpool_limits

SEED = 42
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
FEATURES = [f"emb_{i}" for i in range(150)]
LABEL_RULE = {'minimum_support': 3.0, 'prior_strength': 2.0, 'positive_threshold': 0.6, 'negative_threshold': 0.1, 'maximum_generated_weight': 0.25, 'maximum_class_fraction': 0.5}
PCA_DIMENSIONS = 30
K_GRID = [10,20,30]
COVARIANCE_GRID = ["tied","diag"]
from sklearn.base import clone
from sklearn.decomposition import PCA
cwd = Path.cwd()
DATA = next(q for q in [cwd, cwd/"part1", cwd/"AML-Midterm"/"part1"]
            if (q/"part1_train.csv").is_file() or (q/"data"/"part1_train.csv").is_file()).resolve()
INPUT_DIR = DATA if (DATA/"part1_train.csv").is_file() else DATA/"data"
OUT = DATA/"experiments"/"gmm_pca_grid"
for folder in ["datasets", "label_audits", "clusters", "clustering_models", "classifier_models", "validation_predictions", "submissions", "labeling_models", "clustering_inputs"]:
    (OUT/folder).mkdir(parents=True, exist_ok=True)
train = pd.read_csv(INPUT_DIR/"part1_train.csv")
valid = pd.read_csv(INPUT_DIR/"part1_val.csv")
X = train[FEATURES].to_numpy()
X_valid = valid[FEATURES].to_numpy()
truth = valid[REASONS].to_numpy(dtype=int)
original_y = train[REASONS].to_numpy(dtype=int)
known = train[REASONS].isin([0,1]).all(axis=1).to_numpy()
assert known.sum()==157 and len(train)==1171 and len(valid)==78
assert train.ID.is_unique and valid.ID.is_unique and set(train.ID).isdisjoint(valid.ID)
assert np.isfinite(X).all() and np.isfinite(X_valid).all()
assert np.isin(truth,[0,1]).all()
baseline = joblib.load(DATA/"experiments/kmeans/models/baseline_C1000.joblib")
legacy_packet = joblib.load(DATA/"experiments/kmeans/models/k02.joblib")
classifier_scaler = baseline.named_steps["scaler"]
cluster_scaler = legacy_packet["kmeans_scaler"]
assert classifier_scaler.n_samples_seen_==157 and cluster_scaler.n_samples_seen_==1171
XS = classifier_scaler.transform(X)
VS = classifier_scaler.transform(X_valid)
XC = cluster_scaler.transform(X)
protected_paths = [INPUT_DIR/"part1_train.csv", INPUT_DIR/"part1_val.csv"] + sorted(DATA.glob("*submission.csv")) + [
    DATA/"experiments/kmeans/models/k02.joblib", DATA/"experiments/kmeans/datasets/k02_train.csv"
]
protected_hashes = {str(q.relative_to(DATA)):hashlib.sha256(q.read_bytes()).hexdigest() for q in protected_paths}
print("Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.")

Using 150 features; 157 known-label and 1,014 unlabeled training trials; 78 validation trials.


In [2]:
def label_predictions(probabilities):
    probabilities = np.asarray(probabilities)
    assert probabilities.shape[1]==5 and np.isfinite(probabilities).all()
    assert ((probabilities>=0)&(probabilities<=1)).all()
    prediction=(probabilities>=0.5).astype(int)
    empty=prediction.sum(axis=1)==0
    ids=np.flatnonzero(empty)
    prediction[ids, probabilities[ids].argmax(axis=1)]=1
    return prediction,empty


def score_probabilities(probabilities):
    predicted,empty=label_predictions(probabilities)
    return {"sample_f1":f1_score(truth,predicted,average="samples",zero_division=0),
            "macro_f1":f1_score(truth,predicted,average="macro",zero_division=0),
            "fallback_trials":int(empty.sum()),
            **dict(zip([f"f1_{r}" for r in REASONS],f1_score(truth,predicted,average=None,zero_division=0)))},predicted,empty


def save_predictions(name, probabilities):
    metrics,predicted,empty=score_probabilities(probabilities)
    output=valid[["ID"]+REASONS].rename(columns={r:f"actual_{r}" for r in REASONS}).copy()
    for j,r in enumerate(REASONS):
        output[f"probability_{r}"]=probabilities[:,j]
        output[f"predicted_{r}"]=predicted[:,j]
    output["used_fallback"]=empty
    output.to_csv(OUT/"validation_predictions"/f"{name}.csv",index=False)
    return metrics


DATASETS={}
LABEL_COUNTS=[]
FIT_CACHE={}


def save_dataset(name, targets, weights, probabilities=None, component=None, support=None):
    assert targets.shape==weights.shape==(len(train),5)
    np.testing.assert_array_equal(targets[known],original_y[known])
    np.testing.assert_array_equal(weights[known],np.ones((known.sum(),5)))
    assert np.isin(targets,[-100,0,1]).all() and (weights>=0).all()
    assert np.array_equal(weights>0,targets!=-100)
    frame=train[["ID"]+FEATURES].copy()
    audit=train[["ID"]].copy()
    for j,r in enumerate(REASONS):
        source=np.where(known,"original",np.where(weights[:,j]>0,"generated","unknown"))
        frame[r]=targets[:,j]
        frame[f"weight_{r}"]=weights[:,j]
        audit[r]=targets[:,j]
        audit[f"original_{r}"]=original_y[:,j]
        audit[f"weight_{r}"]=weights[:,j]
        audit[f"label_source_{r}"]=source
        if probabilities is not None:audit[f"estimated_probability_{r}"]=probabilities[:,j]
        pos=(~known)&(targets[:,j]==1)
        neg=(~known)&(targets[:,j]==0)
        LABEL_COUNTS.append({"dataset":name,"reason":r,
            "known_positives":int(original_y[known,j].sum()),
            "known_negatives":int((original_y[known,j]==0).sum()),
            "generated_positives":int(pos.sum()),"generated_negatives":int(neg.sum()),
            "generated_positive_weight":float(weights[pos,j].sum()),
            "generated_negative_weight":float(weights[neg,j].sum())})
    if component is not None:audit["component"]=component
    if support is not None:audit["effective_labeled_support"]=support
    frame["included_for_any_reason"]=(weights>0).any(axis=1)
    frame.to_csv(OUT/"datasets"/f"{name}.csv",index=False)
    audit.to_csv(OUT/"label_audits"/f"{name}.csv",index=False)
    DATASETS[name]={"targets":targets.copy(),"weights":weights.copy(),
                    "generated_trials":int(((weights>0).any(axis=1)&~known).sum())}
    return DATASETS[name]


def partial_labels(responsibilities):
    assert responsibilities.shape[0]==len(train)
    np.testing.assert_allclose(responsibilities.sum(axis=1),1,atol=1e-9)
    component_support=responsibilities[known].sum(axis=0)
    positives=responsibilities[known].T@original_y[known]
    prior=original_y[known].mean(axis=0)
    rate=(positives+LABEL_RULE["prior_strength"]*prior)/(component_support[:,None]+LABEL_RULE["prior_strength"])
    probabilities=responsibilities@rate
    support=responsibilities@component_support
    targets=np.full(original_y.shape,-100,dtype=int)
    weights=np.zeros(original_y.shape,dtype=float)
    targets[known]=original_y[known]
    weights[known]=1.0
    eligible=(~known)&(support>=LABEL_RULE["minimum_support"])
    for j in range(5):
        positive=eligible&(probabilities[:,j]>=LABEL_RULE["positive_threshold"])
        negative=eligible&(probabilities[:,j]<=LABEL_RULE["negative_threshold"])
        npos,nneg=int(positive.sum()),int(negative.sum())
        kp=int(original_y[known,j].sum());kn=int(known.sum()-kp)
        if not npos or not nneg:continue
        fraction=min(LABEL_RULE["maximum_class_fraction"],
                     LABEL_RULE["maximum_generated_weight"]*npos/kp,
                     LABEL_RULE["maximum_generated_weight"]*nneg/kn)
        targets[positive,j]=1;targets[negative,j]=0
        weights[positive,j]=fraction*kp/npos
        weights[negative,j]=fraction*kn/nneg
        assert weights[~known,j].max()<=LABEL_RULE["maximum_generated_weight"]+1e-12
        np.testing.assert_allclose(weights[positive,j].sum()/kp,weights[negative,j].sum()/kn,atol=1e-12)
    components=pd.DataFrame({"component":np.arange(len(component_support)),"effective_labeled_support":component_support,
                             "total_membership_mass":responsibilities.sum(axis=0)})
    for j,r in enumerate(REASONS):
        components[f"weighted_known_positive_{r}"]=positives[:,j]
        components[f"estimated_probability_{r}"]=rate[:,j]
    return targets,weights,probabilities,support,components


def fit_reason_models(dataset_name, kind="logistic", parameter=1000.0):
    dataset=DATASETS[dataset_name]
    estimators=[]
    for j in range(5):
        y=dataset["targets"][:,j];w=dataset["weights"][:,j]
        included=w>0
        assert np.isin(y[included],[0,1]).all() and included[known].all()
        key=(kind,float(parameter),j,hashlib.sha256(y.tobytes()+w.tobytes()).hexdigest())
        if key not in FIT_CACHE:
            if kind=="logistic":
                model=LogisticRegression(C=float(parameter),solver="lbfgs",max_iter=5000,tol=1e-4,random_state=SEED)
                normalized=w[included]*known.sum()/w[included].sum()
                with warnings.catch_warnings():
                    warnings.simplefilter("error",ConvergenceWarning)
                    model.fit(XS[included],y[included],sample_weight=normalized)
            else:
                model={"neighbors":NearestNeighbors(n_neighbors=min(int(parameter),int(included.sum())),algorithm="brute").fit(XS[included]),
                       "target":y[included].copy(),"sample_weight":w[included].copy()}
            FIT_CACHE[key]=model
        estimators.append(FIT_CACHE[key])
    return {"kind":kind,"parameter":float(parameter),"scaler":classifier_scaler,"estimators":estimators,
            "features":FEATURES,"reasons":REASONS,"dataset":dataset_name,
            "logistic_weight_normalization_total":int(known.sum())}


def predict_reason_models(model, raw_features):
    scaled=model["scaler"].transform(np.asarray(raw_features))
    output=[]
    for estimator in model["estimators"]:
        if model["kind"] in ["logistic", "rbf_svm", "gaussian_nb"]:
            positive=list(estimator.classes_).index(1)
            probability=estimator.predict_proba(scaled)[:,positive]
        else:
            distance,indices=estimator["neighbors"].kneighbors(scaled)
            zero=distance==0
            exact=zero.any(axis=1)
            distance_weight=np.divide(1.,distance,out=np.zeros_like(distance),where=distance!=0)
            distance_weight[exact]=zero[exact].astype(float)
            vote=distance_weight*estimator["sample_weight"][indices]
            probability=(vote*estimator["target"][indices]).sum(axis=1)/vote.sum(axis=1)
        output.append(probability)
    probabilities=np.column_stack(output)
    assert np.isfinite(probabilities).all()
    return np.clip(probabilities,0,1)


known_y=np.full_like(original_y,-100);known_y[known]=original_y[known]
known_w=np.zeros(original_y.shape);known_w[known]=1.0
save_dataset("known_only",known_y,known_w)
with threadpool_limits(limits=1):
    reproduced=fit_reason_models("known_only")
np.testing.assert_allclose(predict_reason_models(reproduced,X_valid),baseline.predict_proba(X_valid),rtol=1e-8,atol=1e-10)
reference_rows=[]
for name,model,kaggle in [("submitted_supervised",baseline,0.87072),("submitted_k02",legacy_packet["classifier"],0.87393)]:
    metrics=save_predictions(name,model.predict_proba(X_valid))
    reference_rows.append({"experiment":name,"dataset":"frozen reference","classifier":"logistic","parameter":1000.0,
                           "generated_trials":0 if name=="submitted_supervised" else 388,**metrics,"kaggle_score":kaggle})
print("Confirmed that the new per-reason trainer reproduces the supervised reference.")

Confirmed that the new per-reason trainer reproduces the supervised reference.


In [3]:
SOURCE=DATA/"experiments/method_comparison"
gmm_packet=joblib.load(SOURCE/"clustering_models/gmm_tied_k10.joblib")
gmm_reference=joblib.load(SOURCE/"classifier_models/gmm_tied_k10__logistic_C1000.joblib")
gmm_reference_probability=predict_reason_models(gmm_reference,X_valid)
gmm_reference_metrics=save_predictions("gmm_original_reference",gmm_reference_probability)
np.testing.assert_allclose(gmm_reference_metrics["sample_f1"],0.905982905982906,atol=1e-14)
gmm_reference_frame=pd.read_csv(SOURCE/"datasets/gmm_tied_k10.csv",float_precision="round_trip")
assert gmm_reference_frame.ID.tolist()==train.ID.tolist()
assert gmm_packet["clustering"].n_components==10 and gmm_packet["clustering"].covariance_type=="tied"
assert gmm_packet["label_rule"]==LABEL_RULE
np.testing.assert_array_equal(gmm_packet["scaler"].mean_,cluster_scaler.mean_)
np.testing.assert_array_equal(gmm_packet["scaler"].scale_,cluster_scaler.scale_)
np.testing.assert_array_equal(gmm_reference["scaler"].mean_,classifier_scaler.mean_)
DB_SOURCE=DATA/"experiments/additional_methods"
db_record=json.loads((DB_SOURCE/"submission_manifest.json").read_text())
db_model=joblib.load(DB_SOURCE/db_record["model_file"])
db_metrics=save_predictions("dbscan_current_reference",predict_reason_models(db_model,X_valid))
np.testing.assert_allclose(db_metrics["sample_f1"],0.9102564102564102,atol=1e-14)
for p in [SOURCE/"clustering_models/gmm_tied_k10.joblib",SOURCE/"classifier_models/gmm_tied_k10__logistic_C1000.joblib",
          SOURCE/"datasets/gmm_tied_k10.csv",DB_SOURCE/db_record["model_file"],DB_SOURCE/db_record["submission_file"]]:
    protected_hashes[str(p.relative_to(DATA))]=hashlib.sha256(p.read_bytes()).hexdigest()
PCA_SOURCE=DATA/"experiments/gmm_pca"
pca_packet=joblib.load(PCA_SOURCE/"labeling_models/gmm_tied_k10_pca30.joblib")
pca=pca_packet["pca"]
assert pca.n_components_==30 and pca.n_samples_==1171 and not pca.whiten
np.testing.assert_array_equal(pca_packet["scaler"].mean_,cluster_scaler.mean_)
coordinates=pca.transform(XC)
assert coordinates.shape==(1171,30) and np.isfinite(coordinates).all()
retained_variance=float(pca.explained_variance_ratio_.sum())
pca_control_classifier=joblib.load(PCA_SOURCE/"classifier_models/gmm_tied_k10_pca30.joblib")
pca_control_probability=predict_reason_models(pca_control_classifier,X_valid)
pca_control_metrics=save_predictions("pca30_tied_k10_reference",pca_control_probability)
np.testing.assert_allclose(pca_control_metrics["sample_f1"],0.8974358974358975,atol=1e-14)
pca_control_frame=pd.read_csv(PCA_SOURCE/"datasets/gmm_tied_k10_pca30.csv",float_precision="round_trip")
assert pca_control_frame.ID.tolist()==train.ID.tolist()
for p in [PCA_SOURCE/"labeling_models/gmm_tied_k10_pca30.joblib",PCA_SOURCE/"classifier_models/gmm_tied_k10_pca30.joblib",PCA_SOURCE/"datasets/gmm_tied_k10_pca30.csv"]:
    protected_hashes[str(p.relative_to(DATA))]=hashlib.sha256(p.read_bytes()).hexdigest()
ROWS=[];MODELS={}


def run_gmm(k,covariance):
    name=f"gmm_pca30_{covariance}_k{k}"
    with threadpool_limits(limits=1):
        gmm=clone(gmm_packet["clustering"]).set_params(n_components=k,covariance_type=covariance)
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            gmm.fit(coordinates)
        assert gmm.converged_
        responsibilities=gmm.predict_proba(coordinates)
        y,w,probability,support,components=partial_labels(responsibilities)
        dataset=save_dataset(name,y,w,probability,responsibilities.argmax(axis=1),support)
        is_control=k==10 and covariance=="tied"
        if is_control:
            np.testing.assert_array_equal(y,pca_control_frame[REASONS].to_numpy(dtype=int))
            np.testing.assert_allclose(w,pca_control_frame[[f"weight_{r}" for r in REASONS]].to_numpy(),rtol=1e-12,atol=1e-14)
        components.to_csv(OUT/"clusters"/f"{name}.csv",index=False)
        np.savez_compressed(OUT/"clustering_inputs"/f"{name}.npz",ID=train.ID.to_numpy(dtype=str),coordinates=coordinates,responsibilities=responsibilities)
        packet={"clustering":gmm,"scaler":cluster_scaler,"pca":pca,"family":f"gmm_{covariance}","k":k,
                "label_rule":LABEL_RULE.copy(),"pca_components":30,"clustering_fit_rows":1171}
        joblib.dump(packet,OUT/"labeling_models"/f"{name}.joblib",compress=3)
        classifier=fit_reason_models(name,"logistic",1000)
        classifier["labeling_model_file"]=f"labeling_models/{name}.joblib";classifier["classifier_uses_pca"]=False
        probs=predict_reason_models(classifier,X_valid);metrics=save_predictions(name,probs)
        if is_control:np.testing.assert_allclose(probs,pca_control_probability,rtol=1e-8,atol=1e-10)
        joblib.dump(classifier,OUT/"classifier_models"/f"{name}.joblib",compress=3);MODELS[name]=classifier
        ROWS.append({"experiment":name,"clustering_dimensions":30,"retained_variance":retained_variance,"clusters":k,
                     "covariance_type":covariance,"gmm_iterations":int(gmm.n_iter_),"gmm_converged":bool(gmm.converged_),
                     "generated_trials":dataset["generated_trials"],"generated_reason_slots":int(((w>0)&(~known)[:,None]).sum()),
                     "classifier_dimensions":150,"classifier_C":1000,**metrics,
                     "delta_vs_pca_control":metrics["sample_f1"]-pca_control_metrics["sample_f1"],
                     "delta_vs_original_gmm":metrics["sample_f1"]-gmm_reference_metrics["sample_f1"],
                     "delta_vs_dbscan_leader":metrics["sample_f1"]-db_metrics["sample_f1"],"is_control":is_control,"kaggle_score":np.nan})
        pd.DataFrame(ROWS).to_csv(OUT/"comparison.csv",index=False)
        pd.DataFrame(LABEL_COUNTS).to_csv(OUT/"label_counts.csv",index=False)
        print(f"{name}: sample F1={metrics['sample_f1']:.6f}, macro F1={metrics['macro_f1']:.6f}, "
              f"generated trials={dataset['generated_trials']}, change vs original GMM={ROWS[-1]['delta_vs_original_gmm']:+.6f}, "
              f"vs DBSCAN={ROWS[-1]['delta_vs_dbscan_leader']:+.6f}",flush=True)

print("Frozen PCA30 retained variance:",retained_variance)


Frozen PCA30 retained variance: 0.7706109632972705


In [4]:
for k in K_GRID:run_gmm(k,'tied')


gmm_pca30_tied_k10: sample F1=0.897436, macro F1=0.807576, generated trials=936, change vs original GMM=-0.008547, vs DBSCAN=-0.012821


gmm_pca30_tied_k20: sample F1=0.841880, macro F1=0.708348, generated trials=804, change vs original GMM=-0.064103, vs DBSCAN=-0.068376


gmm_pca30_tied_k30: sample F1=0.893162, macro F1=0.780069, generated trials=647, change vs original GMM=-0.012821, vs DBSCAN=-0.017094


In [5]:
for k in K_GRID:run_gmm(k,'diag')


gmm_pca30_diag_k10: sample F1=0.871795, macro F1=0.768873, generated trials=687, change vs original GMM=-0.034188, vs DBSCAN=-0.038462


gmm_pca30_diag_k20: sample F1=0.822650, macro F1=0.699040, generated trials=812, change vs original GMM=-0.083333, vs DBSCAN=-0.087607


gmm_pca30_diag_k30: sample F1=0.833333, macro F1=0.707118, generated trials=703, change vs original GMM=-0.072650, vs DBSCAN=-0.076923


In [6]:
comparison=pd.DataFrame(ROWS);assert len(comparison)==6
ranked=comparison.sort_values(["sample_f1","is_control","clusters","covariance_type"],ascending=[False,False,True,True],kind="stable")
best=ranked.iloc[0]
selected=joblib.load(OUT/"classifier_models"/f"{best.experiment}.joblib")
selected_probability=predict_reason_models(selected,X_valid)
for row in comparison.itertuples(index=False):
    classifier=joblib.load(OUT/"classifier_models"/f"{row.experiment}.joblib")
    output=pd.read_csv(OUT/"validation_predictions"/f"{row.experiment}.csv",float_precision="round_trip")
    assert output.ID.tolist()==valid.ID.tolist()
    probs=output[[f"probability_{r}" for r in REASONS]].to_numpy()
    predicted=output[[f"predicted_{r}" for r in REASONS]].to_numpy(dtype=int)
    np.testing.assert_allclose(predict_reason_models(classifier,X_valid),probs,rtol=1e-12,atol=1e-12)
    np.testing.assert_array_equal(label_predictions(probs)[0],predicted)
    np.testing.assert_allclose(f1_score(truth,predicted,average="samples",zero_division=0),row.sample_f1,atol=1e-14)
    np.testing.assert_allclose(f1_score(truth,predicted,average="macro",zero_division=0),row.macro_f1,atol=1e-14)
    packet=joblib.load(OUT/"labeling_models"/f"{row.experiment}.joblib")
    scaled=packet["scaler"].transform(X)
    coordinates=packet["pca"].transform(scaled) if packet["pca"] is not None else scaled
    archived=np.load(OUT/"clustering_inputs"/f"{row.experiment}.npz")
    assert archived['ID'].tolist()==train.ID.tolist()
    np.testing.assert_allclose(coordinates,archived['coordinates'],rtol=1e-12,atol=1e-12)
    responsibilities=packet["clustering"].predict_proba(coordinates)
    np.testing.assert_allclose(responsibilities,archived['responsibilities'],rtol=1e-12,atol=1e-12)
    y,w,_,_,_=partial_labels(responsibilities)
    frame=pd.read_csv(OUT/"datasets"/f"{row.experiment}.csv",float_precision="round_trip")
    assert frame.ID.tolist()==train.ID.tolist()
    np.testing.assert_array_equal(y,frame[REASONS].to_numpy(dtype=int))
    np.testing.assert_allclose(w,frame[[f"weight_{r}" for r in REASONS]].to_numpy(),rtol=1e-12,atol=1e-14)
# Independently refit the selected classifier from its exact saved 150-dimensional dataset.
frame=pd.read_csv(OUT/"datasets"/f"{best.experiment}.csv",float_precision="round_trip")
scaled=classifier_scaler.transform(frame[FEATURES].to_numpy())
replay={**selected,"estimators":[]}
with threadpool_limits(limits=1):
    for j,r in enumerate(REASONS):
        y=frame[r].to_numpy(dtype=int);w=frame[f"weight_{r}"].to_numpy();included=w>0
        estimator=clone(selected["estimators"][j])
        with warnings.catch_warnings():
            warnings.simplefilter("error",ConvergenceWarning)
            estimator.fit(scaled[included],y[included],sample_weight=w[included]*157/w[included].sum())
        replay["estimators"].append(estimator)
np.testing.assert_allclose(predict_reason_models(replay,X_valid),selected_probability,rtol=1e-8,atol=1e-10)
np.testing.assert_array_equal(label_predictions(predict_reason_models(replay,X_valid))[0],label_predictions(selected_probability)[0])
for rel,h in protected_hashes.items():assert hashlib.sha256((DATA/rel).read_bytes()).hexdigest()==h
print("PCA30 control, saved label generation, coordinate replay, classifier reloads/rescoring, and independent winner refit verified.")
display(comparison[["experiment","clusters","covariance_type","generated_trials","sample_f1","macro_f1","delta_vs_original_gmm","delta_vs_dbscan_leader"]].round(6))


PCA30 control, saved label generation, coordinate replay, classifier reloads/rescoring, and independent winner refit verified.


,experiment,clusters,covariance_type,generated_trials,sample_f1,macro_f1,delta_vs_original_gmm,delta_vs_dbscan_leader
0,gmm_pca30_tied_k10,10,tied,936,0.897436,0.807576,-0.008547,-0.012821
1,gmm_pca30_tied_k20,20,tied,804,0.841880,0.708348,-0.064103,-0.068376
2,gmm_pca30_tied_k30,30,tied,647,0.893162,0.780069,-0.012821,-0.017094
3,gmm_pca30_diag_k10,10,diag,687,0.871795,0.768873,-0.034188,-0.038462
4,gmm_pca30_diag_k20,20,diag,812,0.822650,0.699040,-0.083333,-0.087607
5,gmm_pca30_diag_k30,30,diag,703,0.833333,0.707118,-0.072650,-0.076923


In [7]:
candidate=None
if best.sample_f1>db_metrics["sample_f1"]+1e-12:
    test=pd.read_csv(INPUT_DIR/"part1_test.csv");sample=pd.read_csv(INPUT_DIR/"part1_sample_submission.csv")
    assert test.ID.is_unique and sample.ID.is_unique and set(test.ID)==set(sample.ID)
    assert set(test.ID).isdisjoint(train.ID) and set(test.ID).isdisjoint(valid.ID)
    prediction,empty=label_predictions(predict_reason_models(selected,test[FEATURES].to_numpy()))
    output=pd.DataFrame({"ID":test.ID,"Predicted":[" ".join(r for r,yes in zip(REASONS,row) if yes) for row in prediction]})
    output=output.set_index("ID").loc[sample.ID].reset_index()
    assert output.shape==(312,2) and output.Predicted.str.len().gt(0).all()
    path=OUT/"submissions"/f"part1_{best.experiment}_logistic_C1000_submission.csv"
    output.to_csv(path,index=False);pd.testing.assert_frame_equal(pd.read_csv(path),output)
    candidate={"experiment":best.experiment,"validation_sample_f1":float(best.sample_f1),"validation_macro_f1":float(best.macro_f1),
               "submission_file":str(path.relative_to(OUT)),"submission_sha256":hashlib.sha256(path.read_bytes()).hexdigest(),
               "model_file":f"classifier_models/{best.experiment}.joblib","model_sha256":hashlib.sha256((OUT/"classifier_models"/f"{best.experiment}.joblib").read_bytes()).hexdigest(),
               "test_rows":312,"test_fallback_trials":int(empty.sum()),"kaggle_score":None,"kaggle_status":"prepared locally; not submitted"}
    (OUT/"submission_manifest.json").write_text(json.dumps(candidate,indent=2)+"\n")
settings={"pca_dimensions":30,"k_grid":K_GRID,"covariance_grid":COVARIANCE_GRID,"configurations":6,
          "pca_reused_from":"experiments/gmm_pca/labeling_models/gmm_tied_k10_pca30.joblib","pca_whiten":False,
          "pca_fit_rows":1171,"cluster_scaler_fit_rows":1171,"classifier_scaler_fit_rows":157,
          "retained_variance":retained_variance,"classifier_features":FEATURES,"classifier_uses_pca":False,"classifier_C":1000,
          "other_gmm_parameters":{key:value for key,value in gmm_packet["clustering"].get_params().items() if key not in ["n_components","covariance_type"]},
          "label_rule":LABEL_RULE,"fitting_weight_total_per_reason":157,"cutoff":0.5,"fallback":True,
          "validation_rows":78,"validation_used_for_fit":False,"test_used_for_fit":False,
          "pca_control_sample_f1":pca_control_metrics["sample_f1"],"pca_control_macro_f1":pca_control_metrics["macro_f1"],
          "reference_gmm_sample_f1":gmm_reference_metrics["sample_f1"],"reference_gmm_macro_f1":gmm_reference_metrics["macro_f1"],
          "reference_dbscan_sample_f1":db_metrics["sample_f1"],"reference_dbscan_macro_f1":db_metrics["macro_f1"],
          "best_grid_experiment":best.experiment,"best_grid_sample_f1":float(best.sample_f1),"best_grid_macro_f1":float(best.macro_f1),
          "leader_changed":bool(best.sample_f1>db_metrics["sample_f1"]+1e-12),"candidate":candidate,
          "protected_input_sha256":protected_hashes,
          "dataset_file_sha256":{str(p.relative_to(OUT)):hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted((OUT/"datasets").glob('*.csv'))},
          "selection":"sample F1; ties prefer previous PCA30 tied/K10 control, then fewer clusters, then covariance string",
          "versions":{"python":platform.python_version(),"sklearn":sklearn.__version__,"numpy":np.__version__,"pandas":pd.__version__},"new_kaggle_scores":None}
(OUT/"settings.json").write_text(json.dumps(settings,indent=2)+"\n")
lines=["# PCA30 GMM cluster-count and covariance sweep","",
       "Six settings cross K=10/20/30 with tied/diagonal covariance. The exact previously fitted training-only PCA30 "
       "and clustering scaler are reused. All other GMM parameters and label-generation rules stay fixed. "
       "The classifier continues to use all 150 original embeddings, its frozen scaler, C=1000, normalized weight "
       "total 157 per reason, cutoff 0.5, and highest-probability fallback.","",
       f"References: previous PCA30 tied/K10 sample F1 **{pca_control_metrics['sample_f1']:.6f}**; "
       f"original no-PCA GMM **{gmm_reference_metrics['sample_f1']:.6f}**; current DBSCAN leader **{db_metrics['sample_f1']:.6f}**.","",
       "| Clusters | Covariance | Generated trials | Sample F1 | Macro F1 | Change vs original GMM | Change vs DBSCAN |",
       "|---:|---|---:|---:|---:|---:|---:|"]
for row in comparison.itertuples(index=False):
    lines.append(f"| {row.clusters} | {row.covariance_type} | {row.generated_trials} | {row.sample_f1:.6f} | {row.macro_f1:.6f} | {row.delta_vs_original_gmm:+.6f} | {row.delta_vs_dbscan_leader:+.6f} |")
lines.extend(["",f"Best grid setting: **{best.experiment}**, sample F1 **{best.sample_f1:.6f}**, macro F1 **{best.macro_f1:.6f}**.","",
       "## Generated-label counts","",
       "| Setting | Reason | Generated positives | Generated negatives | Positive raw weight total | Negative raw weight total |",
       "|---|---|---:|---:|---:|---:|"])
for row in pd.DataFrame(LABEL_COUNTS).query("dataset!='known_only'").itertuples(index=False):
    lines.append(f"| {row.dataset} | {row.reason} | {row.generated_positives} | {row.generated_negatives} | {row.generated_positive_weight:.6f} | {row.generated_negative_weight:.6f} |")
lines.extend(["","## Verification and interpretation","",
       "The PCA30 tied/K10 control reproduces its prior targets, weights, and validation probabilities. All persisted "
       "scaler/PCA/GMM packets replay their exact coordinates, responsibilities, and generated labels/weights. "
       "All classifiers reload and rescore, and the selected classifier independently refits from its saved dataset "
       "with matching probabilities. Known labels and raw weights are preserved. Validation/test never enter fitting. "
       "Protected original artifacts retain their checksums.","",
       "The larger diagonal GMM is inspired by Ani's setup but retains our training-only preprocessing, seed, covariance "
       "regularization, label-generation rules, and classifier. It is not Ani's full transductive self-training ensemble. "
       "The same repeatedly consulted 78 validation trials are used for selection; no independent test improvement is established.","",
       "Exact generated datasets, label audits, cluster statistics, native labeling/classifier models, training coordinates "
       "and responsibilities, validation predictions, comparison.csv, label_counts.csv, and settings.json are saved locally. "
       "No new features, final validation-label refit, Kaggle submission, commit, or tag was used.",""])
lines.append(f"New candidate prepared locally: `{candidate['submission_file']}`. Kaggle score unknown." if candidate else
             "No grid setting improves the current DBSCAN leader; keep the existing leader.")
(OUT/"SUMMARY.md").write_text("\n".join(lines)+"\n")
print("Best grid setting:",best.experiment,"sample F1:",best.sample_f1,"Leader changed:",settings['leader_changed'])
print("Candidate:",candidate)


Best grid setting: gmm_pca30_tied_k10 sample F1: 0.8974358974358975 Leader changed: False
Candidate: None
